# Lab 2 — Predictive Maintenance: Detecting an Air-Leak from the Sensors  🚆🔧
### AI & Data Science with GenAI (Rail) · Your first machine-learning lab

**Welcome to your first ML lab.** In Lab 1 you *explored* a dataset. Today you build a
**machine-learning model** that reads a metro train's compressor sensors and flags an
**air-leak failure**.

The story: a train's **Air Production Unit (APU)** — the compressor that feeds the brakes
and air suspension — sometimes springs an air leak. When it fails, the train is pulled from
service. You have **real sensor readings** with the true failures marked.

> **Be precise about what we're building.** Our model looks at the sensors for a given
> minute and decides if a leak is happening **in that same minute**. That is **detection**
> (spotting a leak in progress) — *not* early warning. Real *prediction* (a warning minutes
> ahead) needs a harder set-up, which we name at the end. Getting this distinction right is
> part of the lesson.

You already know the pandas + plotting from Lab 1. The **new** ideas (features, a train/test
split, training, precision/recall, thresholds) come **one at a time**, each with a worked
example and then a **🔧 Your turn** where *you* do the thinking.

## Step 0 — Get our tools ready and load the data
Same libraries as Lab 1, plus **scikit-learn** — the machine-learning toolkit (new this lab).

In [ ]:
# ============================================================
# STEP 0 : import our tools and load the dataset
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# scikit-learn = the machine-learning toolkit (NEW this lab)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# One row = ONE MINUTE of sensor readings from a real metro train's compressor (APU).
df = pd.read_csv("metropt3_teaching.csv", parse_dates=["timestamp"])

# The 15 sensor columns we will learn from (pressures, temperatures, currents, valves):
SENSORS = ['TP2','TP3','H1','DV_pressure','Reservoirs','Oil_temperature',
           'Motor_current','COMP','DV_eletric','Towers','MPG','LPS',
           'Pressure_switch','Oil_level','Caudal_impulses']

df.head()

## Step 1 — First look
How big is it, and what do the numbers look like? (Same tools as Lab 1.)

In [ ]:
print("Rows and columns:", df.shape)   # ~144,000 minutes, 17 columns
df[SENSORS].describe()

> **Reading it:** each row is one minute. The 15 sensors are the *clues*; the column
> `airleak_failure` is **1** during a real air-leak failure and **0** the rest of the time.

In [ ]:
# 🔧 Your turn (think, don't copy):
# Across the 15 sensors, WHICH sensor has the highest average value?
# Approach: you can get all 15 averages at once with df[SENSORS].mean(),
# then find the largest. Then, in a comment, write the sensor's name.

## Step 2 — How rare are failures?
This is the single most important fact about the problem.

In [ ]:
print(df["airleak_failure"].value_counts())
print("Failure minutes: %.2f%% of all minutes" % (100 * df["airleak_failure"].mean()))

> **Insight:** only about **1.3%** of minutes are failures. The data is **very imbalanced**.
> Keep this in mind — it is exactly why "accuracy" will fool us later.

In [ ]:
# 🔧 Your turn:
# (a) Draw a bar chart of the two counts (normal vs failure) so the imbalance is visible.
# (b) Work out: roughly how many NORMAL minutes are there for every 1 FAILURE minute?
#     (divide the two counts). Print the number.
# Approach: value_counts() gives you both counts; plt.bar(...) draws them.

## Step 3 — SEE a failure in the sensors
Before any modelling, look. We zoom into the real air-leak failure of **18 April 2020** and
plot one sensor. Red dots mark the failure minutes.

In [ ]:
one_day = df[(df["timestamp"] >= "2020-04-17") & (df["timestamp"] <= "2020-04-19")]

plt.figure(figsize=(12, 3))
plt.plot(one_day["timestamp"], one_day["DV_pressure"], lw=0.8)
fail = one_day[one_day["airleak_failure"] == 1]
plt.scatter(fail["timestamp"], fail["DV_pressure"], color="red", s=6, label="failure")
plt.title("DV_pressure around the 18 Apr 2020 air-leak failure")
plt.xlabel("time"); plt.ylabel("DV_pressure"); plt.legend()
plt.tight_layout(); plt.show()

> **Insight:** during the leak, `DV_pressure` jumps to a whole different level — normally it
> sits near 0. That obvious, sustained change is what a model (or even a simple rule) can
> latch onto. Keep that in mind; we come back to it at Step 9.

In [ ]:
# 🔧 Your turn:
# Plot a DIFFERENT sensor (say "Oil_temperature") over the same two-day window.
# Then, in a comment, answer: does THIS sensor also change during the failure,
# or does it look the same as normal?

## Step 4 — Features (X) and target (y)  ⭐ new idea
Machine learning needs two things:
- **X** — the *features*: the columns the model looks at (our 15 sensors).
- **y** — the *target*: the answer we want it to predict (`airleak_failure`).

> ⚠️ Notice X and y are read from the **same minute**. That is why this is **detection**,
> not forecasting — the model has no information from *before* the failure.

In [ ]:
X = df[SENSORS]            # the 15 sensor columns  (the clues)
y = df["airleak_failure"]  # 1 = failure, 0 = normal (the answer)
print("X (features) shape:", X.shape)
print("y (target) failures:", int(y.sum()), "out of", len(y), "minutes")

In [ ]:
# 🔧 Your turn:
# How many individual sensor readings does the model actually look at in total?
# That is: number of rows TIMES number of columns in X.
# Approach: X.shape gives (rows, columns); multiply them (or use X.size).

## Step 5 — Split into TRAIN and TEST — by time  ⭐ new idea
We teach the model on some data (**train**) and check it on data it has never seen
(**test**). For anything time-based we **train on the past and test on the future** — never
let the model peek ahead. We cut at **1 May 2020**.

In [ ]:
cutoff = pd.Timestamp("2020-05-01")
train = df[df["timestamp"] <  cutoff]   # Feb–Apr (contains the 18 Apr failure)
test  = df[df["timestamp"] >= cutoff]   # May onwards (contains the 29–30 May failure)

X_train, y_train = train[SENSORS], train["airleak_failure"]
X_test,  y_test  = test[SENSORS],  test["airleak_failure"]

print("train:", len(train), "rows,", int(y_train.sum()), "failures")
print("test :", len(test),  "rows,", int(y_test.sum()),  "failures")

> **Insight — a caveat you must be honest about:** this teaching dataset contains only
> **two** failure episodes — one in **April**, one in **late May**. Our split learns from the
> April one and is graded on the May one. So a high score means *"the model recognised the
> one May episode"* — **not** proof it works across many trains and many kinds of fault. With
> only two events you **cannot** claim fleet-level performance. Write this in your report.

In [ ]:
# 🔧 Your turn:
# Confirm the split does not overlap in time.
# Print the LAST timestamp in train and the FIRST timestamp in test,
# then, in a comment, state whether they overlap (yes / no).
# Approach: a column's .max() and .min() give the last and first values.

## Step 6 — Train a classifier  ⭐ new idea
A **Random Forest** learns patterns linking the sensor readings to failures, from thousands
of examples. Training is just two lines.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)     # <-- this is the "learning" step
print("Model trained on", len(X_train), "rows.")

In [ ]:
# 🔧 Your turn:
# Use the trained model to make predictions on the TEST set, and print how many
# failure-minutes it flagged. The method that makes predictions is  model.predict(...).
# Then, in a comment: the test set has only 390 real failures — is the model flagging
# about that many, many more, or many fewer? What does the gap suggest?

## Step 7 — Why "accuracy" is the wrong score  ⭐ the key lesson
Because failures are rare, a lazy model that **always says "no failure"** scores ~99%
accuracy — while catching **nothing**. So we compare our model to that do-nothing baseline,
then look at the scores that actually matter.

In [ ]:
pred = model.predict(X_test)
baseline = 1 - y_test.mean()          # accuracy of "always predict no failure"
acc = accuracy_score(y_test, pred)

print(f"Do-nothing baseline accuracy : {baseline:.3f}")
print(f"Our model accuracy           : {acc:.3f}")
print()
print("Confusion matrix  [rows = true 0/1, cols = predicted 0/1]:")
print(confusion_matrix(y_test, pred))

> **Insight:** the do-nothing model has *higher* accuracy than ours — yet catches **zero**
> failures. Accuracy is the wrong score for rare events. Two better scores:
> - **Recall** = of all real failures, how many did we catch? (misses = stranded trains)
> - **Precision** = of all the alarms we raised, how many were real? (low = false alarms)

In [ ]:
# 🔧 Your turn:
# Compute the model's PRECISION and RECALL for the failure class.
# Just like accuracy_score(y_true, y_pred), there are precision_score(...) and
# recall_score(...) that take the same two arguments.
# Then, in a comment: which of the two is the WEAK one here, and why is a low value
# of THAT score a problem for the maintenance team?

## Step 8 — The alarm threshold: a real trade-off  ⭐ new idea
The model doesn't just say yes/no — it gives a **probability** of failure. By default it
alarms at 0.5. **Raising** that threshold means we only alarm when very sure: fewer false
alarms, but we start **missing** real failures. Let's watch the trade-off.

In [ ]:
proba = model.predict_proba(X_test)[:, 1]     # probability of failure for each minute

thresholds = np.linspace(0.1, 0.95, 18)
recalls    = [recall_score(y_test,    (proba >= t).astype(int), zero_division=0) for t in thresholds]
precisions = [precision_score(y_test, (proba >= t).astype(int), zero_division=0) for t in thresholds]

plt.figure(figsize=(9, 4))
plt.plot(thresholds, recalls,    marker="o", label="recall (failures caught)")
plt.plot(thresholds, precisions, marker="o", label="precision (alarms that are real)")
plt.title("Raising the alarm threshold: recall vs precision")
plt.xlabel("Alarm threshold"); plt.ylabel("Score"); plt.legend()
plt.tight_layout(); plt.show()

> **Insight:** at a low threshold we catch almost every failure but cry wolf constantly. Raise
> it and the false alarms fall — but past a point we start **missing** real failures. There is
> **no free lunch**: where you set the threshold is a **business decision**, not a maths one.
> That decision is the heart of your report.

In [ ]:
# 🔧 Your turn:
# At a threshold of 0.90, how many FALSE ALARMS and how many MISSED failures are there?
# Approach: turn the probabilities into 0/1 with (proba >= 0.90), then
#   a false alarm = predicted 1 but truly 0 ;  a miss = predicted 0 but truly 1.
#   (compare against y_test.values so the positions line up.)
# Then, in a comment: compared with the default 0.5 threshold, did raising it HELP or
# HURT, and who benefits (the depot? the passengers?)?

## Step 9 — Do we even need the model? A one-rule baseline  ⭐ think like a skeptic
Before trusting any fancy model, always ask: **could a dumb rule do almost as well?** From
Step 3 we saw `DV_pressure` normally sits near 0 and jumps during a leak. So let's try the
simplest possible detector — *alarm whenever `DV_pressure` is above 0.5* — and compare it to
the Random Forest.

In [ ]:
# The "dumb" one-rule detector — no machine learning at all:
simple_pred = (X_test["DV_pressure"] > 0.5).astype(int)

print("One-rule detector  (DV_pressure > 0.5):")
print("  recall   :", round(recall_score(y_test, simple_pred, zero_division=0), 3))
print("  precision:", round(precision_score(y_test, simple_pred, zero_division=0), 3))
print()
print("Random Forest (default threshold), for comparison:")
print("  recall   :", round(recall_score(y_test, pred, zero_division=0), 3))
print("  precision:", round(precision_score(y_test, pred, zero_division=0), 3))

> **Insight:** the one-line rule catches basically **every** failure too (recall ~0.99), and
> its precision is almost the same as the Random Forest's. In other words, the model is
> **barely better than a single `if` statement** — because this failure is such an obvious,
> sustained change in the sensors. High scores here are **easy, not clever**. The habit to
> keep for life: *always compare a fancy model to a simple baseline before being impressed.*

In [ ]:
# 🔧 Your turn:
# Try the one-rule detector again with a DIFFERENT cutoff (for example DV_pressure > 1.0),
# or on a different sensor from Step 3. Print its recall and precision.
# Then, in a comment: is the Random Forest actually much better than your simple rule here?

## 🏁 Challenge (do BOTH)
1. **Which sensors matter most?** Plot the model's `feature_importances_` for the top ~8
   sensors. Does the top one match the sensor from the **worked** Step 3 example (DV_pressure)?
   *(Hint: `pd.Series(model.feature_importances_, index=SENSORS).sort_values().tail(8).plot(kind="barh")`.)*
2. **Put a price on mistakes.** Say a **false alarm costs 1** (a wasted depot check) and a
   **missed failure costs 50** (a stranded train). Work out the total cost at thresholds
   **0.5, 0.8, 0.9, 0.95** and say which threshold is **cheapest**.
   *(Approach: for each threshold count false alarms and missed like in Step 8, then
   `cost = false_alarms * 1 + missed * 50`.)*

In [ ]:
# 🏁 Challenge — your workspace

# Part 1: which sensors matter most?


# Part 2: cheapest threshold under the given costs

## ✅ Now write your report
You have built a failure **detector**, judged it honestly, and seen the trade-offs. The
**deliverable is your report** (in the Student Report document), written in **your own
words**. Make sure it covers:
- how well the detector works — **and on how little evidence** (only two failure episodes);
- why "accuracy" was misleading here (use the do-nothing baseline);
- the alarm **threshold you would deploy** and the trade-off it accepts (false alarms vs
  missed failures) — use the cost challenge to back it up;
- the honest **limits**: this is **detection, not early warning** (features and label are the
  same minute), a one-rule baseline does nearly as well, and it was trained on one train /
  two failures — so what would you need before trusting it on a fleet?

> ⚠️ **Do not paste code or cell outputs into the report.** Numbers are your *evidence*; the
> report is your *explanation* of what they mean. A report that is just pasted outputs earns
> no marks — the thinking is the point.

Save this notebook (File ▸ Save) and submit it **together with** your written report.

*Data: MetroPT-3 (Metro do Porto APU sensors, 2020). UCI ML Repository, dataset 791.
Licence: Creative Commons Attribution 4.0 (CC BY 4.0) — free to use, with attribution.*